# Лабораторная работа №2. Классификация

Датасет: Wine Quality (красные и белые португальские вина «Vinho Verde»), файл `winequalityN.csv`.
Инструменты: pandas, NumPy, scikit-learn, matplotlib, seaborn, TensorFlow/Keras, TensorBoard.

Ноутбук выполняется сверху вниз. Генератор случайных чисел в скрипте зафиксирован (`random_state=42`), повторный запуск воспроизводит те же разбиение, сетки и метрики.

## 1. Введение

### Цель работы

Изучение методов классификации данных, реализованные в библиотеке Scikit-Learn, а также ознакомление с нейронными сетями с использованием библиотек TensorFlow и TensorBoard.

### Задачи

1. Необходимо выбрать и подготовить датасет для классификации, затем построить классификационные модели с помощью пяти методов:
   1.1 Наивный Байесовский классификатор (GaussianNB, MultinomialNB, ComplementNB, BernoulliNB);
   1.2 Деревья решений (Decision Tree);
   1.3 Линейный дискриминантный анализ (Linear Discriminant Analysis);
   1.4 Метод опорных векторов (Support Vector Machines);
   1.5 Метод ближайших соседей (k-Nearest Neighbors);
2. Затем сравните качество работы классификаторов с помощью следующих метрик:
   2.1 Accuracy (точность),
   2.2. Precision (доля правильно классифицированных положительных примеров),
   2.3 Recall (чувствительность, доля верно найденных положительных примеров),
   2.4 F1-Score (среднее гармоническое precision и recall),
   2.5 Площадь под кривой ROC (AUC-ROC).
3. Настройте гиперпараметры каждого метода и проведите исследование влияния различных параметров на качество классификации.
4. Реализуйте и протестируйте нейронную сеть на TensorFlow, исследуйте эффект настройки гиперпараметров и визуализируйте процесс обучения с помощью инструмента TensorBoard.

**Постановка задачи.** По одиннадцати физико-химическим показателям образца (спирт, плотность, кислотность, сахар, диоксид серы, хлориды и другие) и типу вина (красное или белое) нужно определить, считают ли дегустаторы это вино **хорошим**.

Исходная оценка `quality` — целое число от 3 до 9, медиана оценок экспертов. Задача сделана бинарной:

- класс 1 — **хорошее вино**, `quality >= 7`;
- класс 0 — **обычное вино**, `quality < 7`.

Хороших вин меньше пятой части выборки, поэтому классы несбалансированы, и обычная доля верных ответов (Accuracy) здесь вводит в заблуждение. Оценка качества вина — ещё и субъективное мнение людей, поэтому метки шумные, а потолок качества классификации заведомо ниже, чем на чисто «физических» измерениях.

### Метрики

Положительный класс — «хорошее вино». Через TP, FP, FN, TN обозначены правильно найденные хорошие вина, ложные тревоги, пропущенные хорошие вина и правильно отброшенные обычные.

| Метрика | Формула | Что означает здесь |
|---|---|---|
| Accuracy | \((TP+TN)/(TP+TN+FP+FN)\) | доля верных ответов; при 81% обычных вин «всегда отвечать: обычное» уже даёт 0.81 |
| Precision | \(TP/(TP+FP)\) | какая доля вин, названных хорошими, действительно хорошая |
| Recall | \(TP/(TP+FN)\) | какую долю настоящих хороших вин удалось найти |
| F1-Score | \(2PR/(P+R)\) | среднее гармоническое Precision и Recall |
| AUC-ROC | площадь под ROC-кривой | качество ранжирования вин по вероятности «хорошее», не зависит от порога 0.5 |

Для несбалансированных классов главной сравнительной метрикой выбран **AUC-ROC**. При равенстве, в пределах шума, решают Recall и F1 по хорошему классу.


In [1]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image, Markdown, display

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda v: f"{v:.4f}")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)

# Ноутбук ищет results/ и figures/ рядом с собой (или уровнем выше).
ROOT = Path.cwd()
if not (ROOT / "results").exists() and (ROOT.parent / "results").exists():
    ROOT = ROOT.parent
RESULTS = ROOT / "results"
FIGURES = ROOT / "figures"
LOG_ROOT = Path(r"C:\lr2_wine_tensorboard")

print("Рабочая папка :", ROOT)
print("Таблицы       :", RESULTS, "-", len(list(RESULTS.glob("*.csv"))), "файлов CSV")
print("Графики       :", FIGURES, "-", len(list(FIGURES.glob("*.png"))), "файлов PNG")
print("Журнал TensorBoard:", LOG_ROOT)


def load(name, **kwargs):
    """Читает готовую таблицу из results/."""
    return pd.read_csv(RESULTS / name, **kwargs)


def show_figure(name):
    """Показывает готовый график из figures/ (нужен, если markdown-вставки не отображаются)."""
    display(Image(filename=str(FIGURES / name)))

Рабочая папка : C:\Users\Данил\Desktop\ИИ\2LB_MO
Таблицы       : C:\Users\Данил\Desktop\ИИ\2LB_MO\results - 7 файлов CSV
Графики       : C:\Users\Данил\Desktop\ИИ\2LB_MO\figures - 26 файлов PNG
Журнал TensorBoard: C:\lr2_wine_tensorboard


## 2. Выбор и подготовка датасета

Исходный файл `winequalityN.csv` - объединение двух таблиц: красных и белых вин. В нём 6497 строк. Это сырой кадр: текстовый столбец `type` (`red` / `white`), одиннадцать числовых показателей, оценка `quality` и единичные пропуски в нескольких столбцах. Ниже разобрано, что делает скрипт и зачем.

### Что сделано с таблицей

1. **Тип вина.** Столбец `type` заменён числовым признаком `is_red`: красное = 1, белое = 0. Так он становится обычным бинарным признаком для всех моделей.
2. **Цель.** Из оценки `quality` построена цель `good`: 1, если `quality >= 7`, иначе 0. Сам столбец `quality` после этого убран из признаков: иначе модель получила бы ответ в готовом виде.
3. **Пропуски.** Единичные пропуски в числовых столбцах заполнены медианами столбцов. Медиана устойчива к выбросам, а пропусков так мало, что выбрасывать из-за них строки нет смысла.
4. **Дубликаты.** Точные повторы строк (по всем исходным столбцам вместе с `quality`) удалены. В этом наборе их очень много - **1168**, потому что у разных партий вина одинаковые лабораторные показатели встречаются часто. Если оставить повторы, один и тот же объект может одновременно попасть в обучение и в тест, и оценка на тесте станет завышенной.

### Признаки

После очистки остаётся **12 признаков**: `is_red` и 11 физико-химических показателей.

| Признак | Смысл |
|---|---|
| `fixed_acidity` | нелетучие кислоты (винная, яблочная), г/дм³: «скелет» вкуса |
| `volatile_acidity` | летучая (уксусная) кислота, г/дм³: избыток даёт уксусный запах и считается дефектом |
| `citric_acid` | лимонная кислота, г/дм³: добавляет свежесть |
| `residual_sugar` | остаточный сахар после брожения, г/дм³: сладость |
| `chlorides` | хлориды (соли), г/дм³: солоноватость вкуса |
| `free_sulfur_dioxide` | свободный диоксид серы, мг/дм³: консервант и антиоксидант |
| `total_sulfur_dioxide` | общий диоксид серы (свободный + связанный), мг/дм³ |
| `density` | плотность, г/см³: определяется спиртом и сахаром |
| `pH` | водородный показатель: чем ниже, тем кислее вино |
| `sulphates` | сульфаты (сульфат калия), г/дм³: влияют на уровень SO₂ и стойкость вина |
| `alcohol` | объёмная доля спирта, % |
| `is_red` | тип вина: 1 - красное, 0 - белое |

Масштабы у признаков очень разные: `density` почти всегда около 0.99, `total_sulfur_dioxide` - десятки и сотни, `alcohol` - от 8 до 14. Это важно для всех методов, чувствительных к масштабу: kNN, SVM, LDA и методов Байеса с неотрицательными счётчиками.

### Размер таблицы

Итог очистки: **5329 объектов** и **12 признаков**. Хороших вин (`quality >= 7`) **1011 (18.97%)**, обычных **4318 (81.03%)**. Обычных больше примерно в 4.27 раза.

Чтобы заранее понять, какие признаки отличают хорошее вино от обычного, для каждого признака посчитан размер эффекта Cohen's d: разность средних в двух классах, делённая на объединённое стандартное отклонение. Таблица читается из `results/эффект_признаков.csv`.

In [2]:
effects = load("эффект_признаков.csv")
display(Markdown("### Размер эффекта признаков (Cohen's d, положительный знак - выше у хороших вин)"))
display(effects[["Признак", "Cohen_d", "Среднее обычное", "Среднее хорошее"]])

display(Markdown("### Сводка по очищенной таблице"))
summary = pd.DataFrame(
    {
        "Показатель": ["Объектов после очистки", "Признаков", "Хороших (quality >= 7)", "Обычных", "Удалено дубликатов"],
        "Значение": [5329, 12, 1011, 4318, 1168],
    }
)
display(summary)
print(f"Доля хороших вин: {1011 / 5329:.4f}; отношение обычных к хорошим: {4318 / 1011:.2f}")

### Размер эффекта признаков (Cohen's d, положительный знак - выше у хороших вин)

,Признак,Cohen_d,Среднее обычное,Среднее хорошее
0,alcohol,1.1553,10.3087,11.5727
1,density,-0.7913,0.9950,0.9927
2,chlorides,-0.4783,0.0595,0.0444
3,volatile_acidity,-0.4108,0.3557,0.2943
4,residual_sugar,-0.2384,5.2400,4.2548
5,is_red,-0.2163,0.2721,0.1820
6,total_sulfur_dioxide,-0.1845,116.0396,106.2868
7,citric_acid,0.1681,0.3144,0.3371
8,fixed_acidity,-0.1259,7.2482,7.0795
9,pH,0.1247,3.2207,3.2404


### Сводка по очищенной таблице

,Показатель,Значение
0,Объектов после очистки,5329
1,Признаков,12
2,Хороших (quality >= 7),1011
3,Обычных,4318
4,Удалено дубликатов,1168


Доля хороших вин: 0.1897; отношение обычных к хорошим: 4.27


### Баланс классов

Первый график показывает, сколько в таблице хороших и обычных вин. Это важнее, чем кажется: от баланса зависит, чему верить - Accuracy или Recall.

![Баланс классов](figures/01_баланс_классов.png)

### Распределения признаков по классам

Ящики с усами показывают медиану, квартили и разброс каждого признака отдельно для обычных и хороших вин. Чем сильнее ящики двух классов сдвинуты друг относительно друга, тем больше информации несёт признак.

![Ящики с усами по классам](figures/02_ящики_с_усами.png)

### Корреляции признаков

Тепловая карта показывает попарные корреляции признаков. Часть признаков связана химически: плотность зависит от спирта и сахара, общий диоксид серы включает свободный, тип вина связан с содержанием серы и летучей кислотностью. Эта зависимость нарушает предпосылку независимости в наивном Байесе, а для LDA означает, что ковариационная матрица плохо обусловлена, и сжатие (`shrinkage`) может помочь.

![Корреляционная матрица признаков](figures/03_корреляция_признаков.png)

### Вывод по выбору и подготовке данных

В файле было 6497 строк. Удалено 1168 полных дубликатов, осталось **5329 объектов** и **12 признаков** (`is_red` плюс 11 физико-химических). Хороших вин 1011 (18.97%), обычных 4318 (81.03%). Перекос заметный, около 1 к 4.3: модель, которая всегда отвечает «обычное», получила бы Accuracy около 0.81, поэтому дальше Accuracy читается только вместе с Recall и AUC-ROC.

По размеру эффекта лидирует **alcohol** с \(d = +1.155\): у хороших вин средний градус 11.57% против 10.31% у обычных. Это самый сильный одиночный сигнал: хорошие вина в среднем крепче. На втором месте **density** (\(d = -0.791\)): у хороших вин плотность ниже (0.9927 против 0.9950 г/см³). Это не независимый признак: плотность в первую очередь отражает спирт и сахар, и спирт тянет её вниз. Дальше идут **chlorides** (\(d = -0.478\), 0.0444 против 0.0595: у хороших вин меньше солей) и **volatile_acidity** (\(d = -0.411\), 0.294 против 0.356: у хороших вин меньше уксусной кислоты, то есть меньше дефектов). Остальные признаки слабее: остаточный сахар (\(-0.238\)), тип вина (\(-0.216\): красных среди хороших меньше), общий диоксид серы (\(-0.185\)). Свободный диоксид серы (\(d = +0.032\)) и сульфаты (\(+0.091\)) почти не различают классы по среднему.

Сильные одиночные признаки (спирт, плотность) подсказывают, что даже простая модель найдёт часть сигнала. Но \(|d| = 1.155\) - это умеренно большое, а не огромное разделение, и распределения классов сильно перекрываются. Поэтому потолок AUC-ROC здесь около 0.85, а не около 0.99, как на «физических» задачах. Корреляция `density` со спиртом и сахаром означает, что признаки не независимы: для GaussianNB это повод ждать недооценки, для LDA - повод проверить `shrinkage`.

## 3. Разбиение выборки

Данные делятся один раз: 80% - обучение, 20% - тест. Параметр `stratify=y` сохраняет долю хороших вин в обеих частях. `random_state=42` фиксирует состав групп.

Тестовая часть не участвует ни в каких сетках. Гиперпараметры выбираются только по обучающей части стратифицированной кросс-валидацией на 5 фолдах (`shuffle=True`, тот же seed). В каждом фолде весь конвейер, включая масштабировщик, обучается заново на обучающих фолдах. Среднее признака, посчитанное по проверяемому фолду, так не попадает в модель.

Пять фолдов на 4263 объектах дают около 853 объектов в каждой проверке и около 162 хороших вин. Это достаточно, чтобы AUC-ROC был устойчивым, но различия в третьем знаке между близкими конфигурациями всё равно остаются шумом.

In [3]:
split = load("разбиение_выборки.csv")
display(split)

n_test, n_pos_test = 1066, 202
n_neg_test = n_test - n_pos_test
print(f"Если на тесте всегда отвечать «обычное»: Accuracy = {n_neg_test / n_test:.4f}, Recall = 0, F1 = 0")
print(f"Один неверный объект меняет Accuracy на {1 / n_test:.4f}")
print(f"Одно пропущенное хорошее вино меняет Recall на {1 / n_pos_test:.4f}")

,Выборка,Объектов,Хороших,Обычных,Доля хороших
0,Вся таблица,5329,1011,4318,0.1897
1,Обучение,4263,809,3454,0.1898
2,Тест,1066,202,864,0.1895


Если на тесте всегда отвечать «обычное»: Accuracy = 0.8105, Recall = 0, F1 = 0
Один неверный объект меняет Accuracy на 0.0009
Одно пропущенное хорошее вино меняет Recall на 0.0050


### Вывод по разбиению

В обучении **4263 объекта**, из них хороших **809 (18.98%)**. В тесте **1066 объектов**, хороших **202 (18.95%)**, обычных 864. Доли практически совпадают с исходными 18.97%: стратификация сработала, тест имеет ту же базовую частоту хороших вин.

Тест достаточно большой. Один неверный ответ меняет Accuracy на 0.0009, а одно пропущенное хорошее вино меняет Recall на 0.0050. Это сильно меньше, чем на малых выборках: разницу в 0.01-0.02 по Recall уже нельзя считать «одним пациентом», это 2-4 вина. Но 202 положительных объекта - не так много, поэтому разрыв по Recall в 2-3 пункта всё равно лежит близко к шуму.

Если на тесте всегда отвечать «обычное», то Accuracy будет **0.8105**, Precision не определена (нет ни одной тревоги), Recall и F1 равны нулю. Любая модель, у которой Accuracy ниже 0.81, но Recall заметно выше нуля, поэтому может быть полезнее константы, несмотря на меньшую долю верных ответов. Дальше это важно для SVM.

## 4. Методы классификации

Ниже - идея каждого метода, его предпосылки именно на винных данных и параметры, которые будут перебираться. Один и тот же объект описан вектором \(x\) из 12 признаков (спирт, плотность, летучая кислотность, хлориды, диоксид серы и другие), ответ \(y \in \{0, 1\}\), где 1 - хорошее вино.

### 4.1. Наивный байесовский классификатор

Классификатор выбирает класс с большей апостериорной вероятностью. По формуле Байеса она пропорциональна \(P(y)\,P(x \mid y)\). «Наивность» в том, что признаки считаются условно независимыми при известном классе, и правдоподобие раскладывается в произведение одномерных:

\[
P(y \mid x) \;\propto\; P(y)\prod_{j=1}^{12} P(x_j \mid y).
\]

На винных данных это заведомо приближение: плотность связана со спиртом и сахаром, общий диоксид серы включает свободный. Метод всё равно часто работает, потому что ему нужно верно упорядочить два класса, а не точно восстановить вероятности. Различаются четыре варианта в scikit-learn - тем, как они моделируют \(P(x_j \mid y)\).

**GaussianNB.** Каждый признак в каждом классе описывается нормальным распределением со своими средним и дисперсией. Подходит для непрерывных измерений вроде спирта или pH. Параметр `var_smoothing` добавляет к дисперсиям долю максимальной дисперсии, чтобы не делить на ноль и сгладить слишком узкие гауссианы. Значение по умолчанию - \(10^{-9}\). Перебирается от \(10^{-12}\) до 1 на логарифмической сетке, на сырых и на стандартизованных признаках.

**MultinomialNB.** Признаки трактуются как неотрицательные «счётчики» (частоты слов в тексте). Модель оценивает вероятность каждого признака в классе и умножает вклад на значение признака. На винных данных признаки - не счётчики, но после `MinMaxScaler` они лежат в \([0, 1]\), и модель применима как формальный эксперимент. Параметр `alpha` - сглаживание Лапласа. Сетка: от 0.001 до 10.

**ComplementNB.** Вариант Multinomial, который оценивает вес признака по дополнению класса (по всем остальным объектам). Создан для несбалансированных текстовых классов. Здесь это естественный кандидат из-за перекоса 1 к 4.3, но при двух классах «дополнение» одного класса - это другой класс, поэтому ожидаем почти то же ранжирование, что и у Multinomial.

**BernoulliNB.** Каждый признак превращается в бинарный (больше порога или нет), модель учитывает и наличие, и отсутствие признака. Для винных данных признаки сначала приводятся к \([0, 1]\), затем бинаризуются порогом (сетка 0.0-0.8, значение по умолчанию 0.5). Бинаризация выбрасывает всё, кроме знака отклонения: «спирт выше середины шкалы» вместо точного градуса. Параметры: порог бинаризации и `alpha`.

### 4.2. Дерево решений

Дерево строит последовательность вопросов вида «alcohol > 10.8?» и в каждом узле выбирает вопрос, сильнее всего уменьшающий неоднородность классов. Мера неоднородности - критерий Джини или энтропия:

\[
G = 1 - \sum_k p_k^2, \qquad H = -\sum_k p_k \log_2 p_k.
\]

Дерево не нуждается в масштабировании: разделяющие пороги инвариантны к монотонным преобразованиям признака. Ему безразличны и сильно коррелированные признаки, плотность и спирт просто становятся конкурирующими кандидатами для расщепления. Платой служит склонность к переобучению: неограниченное дерево доводит листья до чистоты и запоминает шум в оценках дегустаторов. Поэтому перебираются `max_depth` (1-12 и без ограничения), `min_samples_leaf` (1, 5, 10, 20) и критерий (`gini`, `entropy`). Ожидание - мелкое дерево с крупными листьями.

### 4.3. Линейный дискриминантный анализ (LDA)

LDA предполагает, что в каждом классе признаки распределены нормально, причём с **одной общей** ковариационной матрицей \(\Sigma\). Тогда граница между классами линейна, а решающее правило - знак выражения

\[
w^\top x + b, \qquad w = \Sigma^{-1}(\mu_1 - \mu_0).
\]

Вектор \(w\) направлен вдоль разности средних классов с поправкой на корреляции признаков. Для винных данных это значит, что LDA сам учтёт связь плотности и спирта. Но когда признаки сильно коррелируют, оценка \(\Sigma\) нестабильна. Лекарство - сжатие (`shrinkage`) ковариационной матрицы к диагональной: оно работает только с солвером `lsqr` (и `eigen`). Перебираются солвер `svd` (по умолчанию, без сжатия) и `lsqr` с `shrinkage` из \(\{0, 0.1, 0.25, 0.5, 0.75, 1, \text{auto}\}\). Входы стандартизуются.

### 4.4. Метод опорных векторов (SVM)

SVM ищет разделяющую поверхность с максимальным зазором между классами. Зазор определяют только объекты на границе или внутри неё - опорные векторы. Параметр `C` задаёт штраф за попадание объекта на «чужую» сторону: большой `C` строит узкий зазор и подгоняется под обучение, малый - допускает больше нарушений и даёт более гладкую границу.

Нелинейная граница получается с ядром. **Линейное ядро** даёт плоскость в пространстве 12 признаков. **RBF-ядро**

\[
K(x, x') = \exp\!\bigl(-\gamma\,\lVert x - x' \rVert^2\bigr)
\]

даёт гибкую границу; `gamma` - обратный радиус влияния одного объекта: малый `gamma` - почти линейная гладкая граница, большой - «острова» вокруг отдельных вин. SVM чувствителен к масштабу: без стандартизации `total_sulfur_dioxide` (сотни) задавил бы `density` (сотые доли). Дополнительный параметр `class_weight="balanced"` увеличивает штраф за ошибки на редком классе в \(n / (2 n_k)\) раз и сдвигает границу в сторону большего Recall. Сетка: `C` от 0.01 до 100, `gamma` в \(\{0.01, 0.1, 1, \text{scale}\}\), с весами и без.

### 4.5. Метод k ближайших соседей (kNN)

kNN вообще ничего не обучает: он запоминает обучающую выборку, а новое вино относит к классу, который чаще встречается среди его \(k\) ближайших соседей. Расстояние - евклидово или манхэттенское:

\[
d_2(x, x') = \sqrt{\sum_j (x_j - x'_j)^2}, \qquad d_1(x, x') = \sum_j |x_j - x'_j|.
\]

Метод очень зависит от масштаба: без стандартизации ближайшие соседи определялись бы почти только диоксидом серы. Малое \(k\) даёт рваную границу и переобучение (при \(k = 1\) каждое вино - «своё»), большое - сглаживает до класса большинства. Вес `distance` придаёт ближним соседям больший вклад и делает вероятности менее ступенчатыми, а это важно для AUC-ROC. Перебираются \(k\) от 1 до 25, веса `uniform` и `distance`, метрики `euclidean` и `manhattan`.

## 5. Настройка гиперпараметров

Каждый подраздел устроен одинаково. Сначала на обучающей выборке считается сетка, затем строится график зависимости качества от параметра, затем формулируется вывод: где качество стоит, где падает и какую конфигурацию забираем в финальное сравнение. Числа в выводах - средние по тем же пяти фолдам (CV AUC-ROC). Тестовая выборка по-прежнему не используется.

Точка сравнения («по умолчанию») у каждого метода - стандартная конфигурация scikit-learn с предобработкой, подходящей методу. Прирост считается как разность CV AUC-ROC выбранной конфигурации и этой стартовой точки.

### 5.1. GaussianNB

Перебирается `var_smoothing` от \(10^{-12}\) до 1 на логарифмической сетке (13 значений), отдельно на сырых признаках и после `StandardScaler`. Значение по умолчанию в scikit-learn - \(10^{-9}\). Ожидание: на левом краю добавка к дисперсии ничтожна, и модель живёт за счёт выборочных средних и дисперсий; на правом краю дисперсии раздуты настолько, что классы перекрываются. Стандартизация должна сделать общую добавку `var_smoothing` одинаково честной для признаков разного масштаба: для `density` (разброс сотые доли) и `total_sulfur_dioxide` (разброс десятки).

![GaussianNB: зависимость CV AUC-ROC от var_smoothing](figures/04_gaussian_nb.png)

#### Вывод по GaussianNB

Стартовая точка (`StandardScaler`, `var_smoothing=1e-9`) даёт CV AUC-ROC **0.7767**. Выбрана конфигурация `StandardScaler`, `var_smoothing=1`: CV AUC-ROC **0.7881**, прирост **+0.0114**. Выбранное значение лежит на правом краю сетки, то есть на «максимально сглаженной» точке. Для винных данных это осмысленно: признаки коррелируют (плотность - спирт - сахар), реальные распределения несимметричны, а большая добавка к дисперсии делает гауссианы шире и снижает самоуверенность модели, которая из-за зависимых признаков считает один и тот же сигнал несколько раз.

Прирост чуть выше 0.01 - небольшой. GaussianNB остаётся одним из самых слабых методов этой работы: нарушенная независимость не лечится одним параметром сглаживания.

### 5.2. MultinomialNB

Все 12 признаков неотрицательны и в исходном масштабе, поэтому модель применима напрямую, но масштабы у них сильно разные. Перебираются два режима: исходный масштаб и `MinMaxScaler`. Сетка `alpha`: 0.001, 0.01, 0.1, 0.5, 1, 2, 5, 10. Стартовая точка - `MinMaxScaler`, `alpha=1`. Ожидание: модель не создана для непрерывных измерений, поэтому качество будет заметно ниже, чем у гауссовского варианта, а `alpha` почти ничего не изменит, так как значения в \([0, 1]\) задают «веса» признаков, а не частоты.

![MultinomialNB: зависимость CV AUC-ROC от alpha](figures/05_multinomial_nb.png)

#### Вывод по MultinomialNB

Выбрана стартовая конфигурация `MinMaxScaler`, `alpha=1`: CV AUC-ROC **0.7414**, прирост **+0.0000**. Сетка по `alpha` не нашла ничего лучше: сглаживание Лапласа на признаках из \([0, 1]\) практически не влияет на ранжирование. Качество существенно ниже гауссовского варианта (0.7881) - это ожидаемая цена за формальное применение «текстовой» модели к непрерывным химическим данным.

Дальше, на тесте, будет видно ещё одно следствие: порог 0.5 у этой модели оказывается таким, что она почти никогда не называет вино хорошим.

### 5.3. ComplementNB

Та же сетка `alpha` и тот же `MinMaxScaler`, что и для MultinomialNB. ComplementNB оценивает веса признаков по дополнению класса, чтобы смягчить перекос в сторону большого класса. На несбалансированных текстах это помогает. При двух классах дополнение одного класса совпадает с другим классом, поэтому ожидаем практически то же ранжирование, что у Multinomial, но с другим порогом решений.

![ComplementNB: зависимость CV AUC-ROC от alpha](figures/06_complement_nb.png)

#### Вывод по ComplementNB

Выбрана `MinMaxScaler`, `alpha=1`: CV AUC-ROC **0.7414**, прирост **+0.0000**. Значение совпадает с MultinomialNB до четвёртого знака, как и предсказывали: при двух классах ранжирование объектов у двух моделей по сути одно и то же. Различие между ними не в AUC, а в положении порога. Это подтвердится на тесте: AUC-ROC у обоих **0.7164**, а Recall у Complement **0.7327** против нуля у Multinomial.

### 5.4. BernoulliNB

У BernoulliNB два параметра: порог бинаризации (0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8 после `MinMaxScaler`) и `alpha` (0.001-10). По умолчанию порог 0.5 и `alpha=1`. На винных данных порог означает «признак выше или ниже такой доли своего диапазона». Низкий порог оставляет почти всё единицами и теряет информацию, высокий - почти всё нулями. Два графика: сначала зависимость от порога, затем от `alpha`.

**Зависимость от порога бинаризации**

![BernoulliNB: порог бинаризации](figures/07_bernoulli_порог.png)

**Зависимость от alpha**

![BernoulliNB: alpha](figures/08_bernoulli_alpha.png)

#### Вывод по BernoulliNB

Стартовая точка (`MinMaxScaler`, порог 0.5, `alpha=1`) даёт CV AUC-ROC **0.7605**. Выбрана конфигурация с порогом **0.4** и `alpha=10`: **0.7702**, прирост **+0.0097**, чуть меньше условной отметки «заметно» в 0.01. Порог сдвинулся ниже середины диапазона: для признаков вроде спирта и плотности информативнее граница, разделяющая «заметно выше минимума», чем строго середина шкалы. `alpha=10` - правый край сетки: сильное сглаживание компенсирует грубость бинаризации.

Бинарные признаки выбрасывают почти всю количественную информацию: «спирт выше порога» вместо точного градуса. Поэтому BernoulliNB уступает GaussianNB (0.7881) и остаётся на нижней части рейтинга.

### 5.5. Дерево решений

Сетка: `max_depth` \(\in \{1, 2, 3, 4, 5, 6, 8, 10, 12, \text{None}\}\), `min_samples_leaf` \(\in \{1, 5, 10, 20\}\), критерий `gini` или `entropy` - 80 комбинаций. Стартовая точка - библиотечная: без ограничения глубины, лист из одного объекта, критерий Джини. Ожидание: неограниченное дерево запомнит шум в оценках дегустаторов и покажет низкий AUC-ROC, а мелкое дерево с крупными листьями будет заметно лучше.

![Дерево: CV AUC-ROC в зависимости от глубины](figures/09_дерево_глубина.png)

#### Вывод по дереву

Стартовая конфигурация (без ограничения глубины, `leaf=1`, `gini`) даёт всего **0.6494**: дерево доводит листья до чистоты и запоминает шум, так что на проверочных фолдах оно ранжирует вина лишь немногим лучше случайного угадывания. Выбрана конфигурация `depth=3`, `leaf=10`, `entropy`: **0.8029**, прирост **+0.1535** - самый большой из всех методов.

Глубина 3 означает не более 8 листьев. Этого достаточно, чтобы взять главные признаки (спирт, летучую кислотность, плотность), но не хватает, чтобы запоминать шум. Крупные листья (не меньше 10 вин) делают долю хороших вин в листе устойчивой, а AUC-ROC зависит именно от этих оценок. Это учебный пример: гиперпараметры дерева важнее выбора самого метода.

Но мелкое дерево даёт всего несколько различных значений вероятности, поэтому ROC-кривая получается ломаной с малым числом точек, а порог 0.5 редко срабатывает на редком классе. На тесте это проявится низким Recall.

### 5.6. LDA

Перебираются солвер `svd` (по умолчанию, без сжатия) и `lsqr` с `shrinkage` \(\in \{0, 0.1, 0.25, 0.5, 0.75, 1, \text{auto}\}\), признаки всегда стандартизуются. Ожидание: корреляции между признаками (плотность - спирт - сахар) делают оценку ковариационной матрицы чувствительной, но 4263 объекта на 12 признаков - много, поэтому сжатие даст разве что небольшой эффект.

![LDA: CV AUC-ROC в зависимости от shrinkage](figures/10_lda_shrinkage.png)

#### Вывод по LDA

Стартовая точка (`svd`, без сжатия) даёт CV AUC-ROC **0.8276**, это уже очень сильный результат для линейного метода. Выбрана конфигурация `lsqr`, `shrinkage=0.1`: **0.8251**, то есть **-0.0025** от старта. Формально выбранная точка хуже стартовой на две с половиной тысячных, но это разница внутри шума: по правилу выбора победил более простой вариант в плоской области, а `svd` лежит в той же полке. Практический вывод - сжатие не нужно: объектов много, ковариационная матрица оценивается устойчиво, и LDA работает хорошо «из коробки».

Важно, что линейный метод с одним проходом (LDA) даёт CV AUC 0.8251, а гибкие SVM и kNN - около 0.846 и 0.848. Граница между хорошим и обычным вином лишь слабо нелинейна.

### 5.7. SVM

Для линейного ядра перебирается `C` от 0.01 до 100, с весами классов и без. Для RBF-ядра - сетка `C` \(\times\) `gamma`: `C` \(\in \{0.01, 0.1, 1, 10, 100\}\), `gamma` \(\in \{0.01, 0.1, 1, \text{scale}\}\), опять с весами `balanced` и без. Все признаки стандартизуются. Стартовая точка - RBF с `C=1`, `gamma="scale"`, без весов. Ожидание: RBF с подобранным `gamma` обойдёт линейное ядро, а веса классов повлияют скорее на порог, чем на AUC-ROC.

**Линейное ядро**

![SVM с линейным ядром](figures/11_svm_линейное_ядро.png)

**RBF-ядро: тепловые карты по C и gamma**

![SVM с RBF-ядром](figures/12_svm_rbf.png)

#### Вывод по SVM

Стартовая конфигурация (RBF, `C=1`, `gamma=scale`, без весов) даёт CV AUC-ROC **0.8206**. Выбрана `rbf`, `C=10`, `gamma=0.01`, веса `balanced`: **0.8458**, прирост **+0.0252**. Выбранная точка - довольно гладкая граница: малое `gamma = 0.01` (минимальное в сетке) делает радиус влияния одного вина большим, а умеренно большое `C = 10` позволяет этой гладкой границе точно подогнаться под данные. Это вариант «почти линейной, но чуть изогнутой» поверхности, что согласуется с выводом по LDA.

Веса `balanced` практически не меняют ранжирование, но сильно двигают порог решения: модель будет называть хорошими заметно больше вин. Цена - много ложных тревог и низкая Accuracy на тесте, а выигрыш - высокий Recall. Это основная причина, почему SVM будет выделяться в финальном сравнении.

### 5.8. kNN

Перебираются \(k\) от 1 до 25, веса `uniform` и `distance`, метрики `euclidean` и `manhattan` - всего 100 комбинаций, признаки стандартизуются. Стартовая точка - \(k=5\), `uniform`, `euclidean`. Ожидание: малое \(k\) переобучается и даёт низкий AUC-ROC, с ростом \(k\) качество растёт и выходит на плато, а вес `distance` помогает сделать вероятности менее ступенчатыми.

![kNN: CV AUC-ROC и Recall в зависимости от k](figures/13_knn.png)

#### Вывод по kNN

Стартовая конфигурация (\(k=5\), `uniform`, `euclidean`) даёт CV AUC-ROC **0.8008**. Выбрана \(k = 23\), `distance`, `manhattan`: **0.8481**, прирост **+0.0474**. Выбранное \(k\) находится почти на правом краю сетки: для шумных меток дегустаторов нужно усреднять по многим соседям, а при малом \(k\) вероятности принимают лишь значения 0, 0.2, 0.4 и так далее, и ранжирование огрублено. Манхэттенская метрика устойчивее к выбросам по отдельным признакам (диоксид серы, хлориды), а вес `distance` сглаживает ступенчатые оценки.

Итог настройки - **лучший CV AUC-ROC среди всех восьми методов**: 0.8481, чуть выше SVM (0.8458).

## Сводка настройки

Теперь все восемь методов вместе: что давала настройка относительно стартовой точки. Таблица читается из `results/настройка_гиперпараметров.csv`, график - столбчатая диаграмма прироста CV AUC-ROC.

In [4]:
tuning = load("настройка_гиперпараметров.csv")
display(Markdown("### Настройка гиперпараметров: до и после (CV AUC-ROC, 5 фолдов)"))
display(tuning.set_index("Метод"))

gains = tuning.set_index("Метод")["Прирост AUC"].sort_values(ascending=False)
print("Прирост по убыванию:")
print(gains.to_string())

### Настройка гиперпараметров: до и после (CV AUC-ROC, 5 фолдов)

,Конфигурация по умолчанию,AUC по умолчанию,Выбранная конфигурация,AUC после выбора,Прирост AUC
Метод,,,,,
GaussianNB,"StandardScaler, var_smoothing=1e-9",0.7767,"StandardScaler, var_smoothing=1",0.7881,0.0114
MultinomialNB,"MinMaxScaler, alpha=1",0.7414,"MinMaxScaler, alpha=1",0.7414,0.0000
ComplementNB,"MinMaxScaler, alpha=1",0.7414,"MinMaxScaler, alpha=1",0.7414,0.0000
BernoulliNB,"MinMaxScaler, порог=0.5, alpha=1",0.7605,"MinMaxScaler, порог=0.4, alpha=10",0.7702,0.0097
Дерево,"depth=None, leaf=1, gini",0.6494,"depth=3, leaf=10, entropy",0.8029,0.1535
LDA,"svd, без shrinkage, StandardScaler",0.8276,"lsqr, shrinkage=0.1, StandardScaler",0.8251,-0.0025
SVM,"StandardScaler, rbf, C=1, gamma=scale, обычные веса",0.8206,"rbf, C=10, gamma=0.01, веса=balanced, StandardScaler",0.8458,0.0252
kNN,"StandardScaler, k=5, uniform, euclidean",0.8008,"k=23, distance, manhattan, StandardScaler",0.8481,0.0474


Прирост по убыванию:
Метод
Дерево           0.1535
kNN              0.0474
SVM              0.0252
GaussianNB       0.0114
BernoulliNB      0.0097
MultinomialNB    0.0000
ComplementNB     0.0000
LDA             -0.0025


![Прирост CV AUC-ROC после настройки](figures/14_прирост_настройки.png)

### Вывод по настройке в целом

Заметный прирост (0.01 и выше) получен у четырёх методов: **Дерево (+0.1535)**, **kNN (+0.0474)**, **SVM (+0.0252)** и **GaussianNB (+0.0114)**. BernoulliNB (+0.0097) почти дотянул до этой отметки. У **MultinomialNB и ComplementNB прироста нет (+0.0000)**: на признаках из \([0, 1]\) сглаживание ничего не меняет. У **LDA** выбранная точка на 0.0025 хуже стартовой (0.8251 против 0.8276), это шум плоской вершины, а не поломка.

Прирост от 0.01 говорит о неудачной стартовой точке. У дерева это слишком мелкие листья (0.6494 против 0.8029), у kNN - слишком малое \(k\) и «мягкие» евклидовы расстояния, у SVM - лишь умеренный выигрыш за счёт `gamma` и весов классов. Околонулевой прирост означает, что стартовая точка уже на верхней полке (LDA), а у наивных мультиномиальных вариантов, что модели просто плохо подходят к непрерывным данным.

Выбранные конфигурации, уходящие в финальное сравнение, по убыванию CV AUC-ROC:

| Метод | Конфигурация | CV AUC-ROC |
|---|---|---|
| kNN | k=23, distance, manhattan, StandardScaler | 0.8481 |
| SVM | rbf, C=10, gamma=0.01, веса=balanced, StandardScaler | 0.8458 |
| LDA | lsqr, shrinkage=0.1, StandardScaler | 0.8251 |
| Дерево | depth=3, leaf=10, entropy | 0.8029 |
| GaussianNB | StandardScaler, var_smoothing=1 | 0.7881 |
| BernoulliNB | MinMaxScaler, порог=0.4, alpha=10 | 0.7702 |
| ComplementNB | MinMaxScaler, alpha=1 | 0.7414 |
| MultinomialNB | MinMaxScaler, alpha=1 | 0.7414 |

Эти AUC слегка оптимистичны: точка выбрана как лучшая на той же кросс-валидации, по которой оценена. Честная проверка - отложенный тест следующего раздела.

## 6. Сравнительный анализ

Выбранные конвейеры заново обучаются на всей обучающей выборке (4263 объекта) и один раз применяются к тесту (1066 объектов). Для SVM на этом шаге включается `probability=True`: ROC и порог решения используют вероятности. Сетки не пересматриваются.

Правило итогового выбора, зафиксированное до просмотра теста:

1. Ориентир ранжирования - AUC-ROC на тесте.
2. Если несколько методов отстают от лидера не больше чем на 0.01, между ними решают Recall хорошего класса, затем F1: для задачи «найти хорошие вина» пропуск хорошего образца важнее, чем лишняя проверка обычного.
3. Более простая модель заменяет лидера, только если она не хуже по полноте и при этом заметно проще (например, LDA вместо SVM или kNN).

Из-за перекоса классов Accuracy в правило не входит: при Accuracy 0.81 константный ответ «обычное» выглядит прилично, а полезной модели не даёт.

In [5]:
test = load("метрики_тест.csv").set_index("Метод")
display(Markdown("### Метрики на тестовой выборке (отсортировано по AUC-ROC)"))
display(test.sort_values(["AUC-ROC", "Recall", "F1-Score"], ascending=False))

# Матрицы ошибок восстанавливаются по Precision, Recall и размерам классов теста.
N_POS, N_NEG = 202, 864
rows = []
for name, r in test.iterrows():
    tp = int(round(r["Recall"] * N_POS))
    fn = N_POS - tp
    fp = int(round(tp / r["Precision"])) - tp if r["Precision"] > 0 else 0
    tn = N_NEG - fp
    rows.append(
        {"Метод": name, "TP": tp, "FN": fn, "FP": fp, "TN": tn, "Названо хорошими": tp + fp, "Accuracy (проверка)": (tp + tn) / (N_POS + N_NEG)}
    )
display(Markdown("### Матрицы ошибок (TP/FN/FP/TN), восстановленные по таблице метрик"))
display(pd.DataFrame(rows).set_index("Метод"))

### Метрики на тестовой выборке (отсортировано по AUC-ROC)

,Accuracy,Precision,Recall,F1-Score,AUC-ROC,CV AUC-ROC
Метод,,,,,,
kNN,0.8340,0.6033,0.3614,0.4520,0.8345,0.8481
SVM,0.7298,0.3961,0.8119,0.5325,0.8331,0.8458
LDA,0.8218,0.5476,0.3416,0.4207,0.8080,0.8251
Дерево,0.8227,0.5802,0.2327,0.3322,0.7855,0.8029
GaussianNB,0.7964,0.4634,0.4703,0.4668,0.7581,0.7881
BernoulliNB,0.7983,0.4235,0.1782,0.2509,0.7559,0.7702
ComplementNB,0.5788,0.2726,0.7327,0.3973,0.7164,0.7414
MultinomialNB,0.8105,0.0000,0.0000,0.0000,0.7164,0.7414


### Матрицы ошибок (TP/FN/FP/TN), восстановленные по таблице метрик

,TP,FN,FP,TN,Названо хорошими,Accuracy (проверка)
Метод,,,,,,
kNN,73,129,48,816,121,0.8340
SVM,164,38,250,614,414,0.7298
LDA,69,133,57,807,126,0.8218
Дерево,47,155,34,830,81,0.8227
GaussianNB,95,107,110,754,205,0.7964
BernoulliNB,36,166,49,815,85,0.7983
ComplementNB,148,54,395,469,543,0.5788
MultinomialNB,0,202,0,864,0,0.8105


### ROC-кривые

ROC-кривая показывает, как меняются доля найденных хороших вин (Recall) и доля ложных тревог при сдвиге порога. Чем кривая выше и левее, тем лучше ранжирование. AUC-ROC - площадь под ней.

![ROC-кривые восьми методов на тесте](figures/15_roc_кривые.png)

### Столбчатые диаграммы метрик

Те же пять метрик по методам рядом: так видно, что лидер по AUC-ROC не обязательно лидер по Recall или F1.

![Метрики восьми методов на тесте](figures/16_столбчатые_метрики.png)

### Матрицы ошибок

Матрицы ошибок на тесте показывают, где именно каждая модель ошибается: пропускает хорошие вина (FN) или называет хорошими обычные (FP).

![Матрицы ошибок на тесте](figures/17_матрицы_ошибок.png)

In [6]:
params = load("параметры_моделей.csv")
display(Markdown("### Параметры итоговых моделей"))
display(params.set_index("Метод"))

# Сравнение CV и теста: насколько оптимистична кросс-валидация.
cmp = test[["CV AUC-ROC", "AUC-ROC"]].copy()
cmp["Разность CV - тест"] = cmp["CV AUC-ROC"] - cmp["AUC-ROC"]
display(Markdown("### CV AUC-ROC и AUC-ROC на тесте"))
display(cmp)

### Параметры итоговых моделей

,Параметры
Метод,
kNN,"k=23, distance, manhattan, StandardScaler"
SVM,"rbf, C=10, gamma=0.01, веса=balanced, StandardScaler"
LDA,"lsqr, shrinkage=0.1, StandardScaler"
Дерево,"depth=3, leaf=10, entropy"
GaussianNB,"StandardScaler, var_smoothing=1"
BernoulliNB,"MinMaxScaler, порог=0.4, alpha=10"
ComplementNB,"MinMaxScaler, alpha=1"
MultinomialNB,"MinMaxScaler, alpha=1"


### CV AUC-ROC и AUC-ROC на тесте

,CV AUC-ROC,AUC-ROC,Разность CV - тест
Метод,,,
kNN,0.8481,0.8345,0.0136
SVM,0.8458,0.8331,0.0127
LDA,0.8251,0.8080,0.0171
Дерево,0.8029,0.7855,0.0174
GaussianNB,0.7881,0.7581,0.0300
BernoulliNB,0.7702,0.7559,0.0143
ComplementNB,0.7414,0.7164,0.0250
MultinomialNB,0.7414,0.7164,0.0250


### Вывод по сравнению методов

**Константа.** Ответ «всегда обычное» дал бы на тесте Accuracy 0.8105 и Recall 0. Это важный ориентир: MultinomialNB как раз и попадает на эту полку, его Accuracy равна **0.8105**, а Precision, Recall и F1 равны **0**. Модель не назвала хорошим ни одного вина, ранжирование (AUC-ROC 0.7164) у неё есть, но порог 0.5 бесполезен. Одна только Accuracy такую модель не отличила бы от константы.

**Лидеры по AUC-ROC.** Первое место у **kNN**: AUC-ROC **0.8345**. Второе - **SVM** (**0.8331**), отставание всего 0.0014. Дальше с большим провалом идут LDA (0.8080), дерево (0.7855), GaussianNB (0.7581), BernoulliNB (0.7559) и два мультиномиальных варианта (0.7164). В полосу «не хуже лидера более чем на 0.01» входят только kNN и SVM. AUC-ROC этих двух моделей различаются на полторы тысячных: для 1066 объектов с 202 хорошими это шум, а не победа.

**Порог решения решает всё остальное.** При почти равном AUC-ROC kNN и SVM принимают противоположные решения:

| Метод | Accuracy | Precision | Recall | F1 | AUC | Найдено хороших | Ложных тревог |
|---|---|---|---|---|---|---|---|
| kNN | 0.8340 | 0.6033 | 0.3614 | 0.4520 | 0.8345 | 73 из 202 | 48 |
| SVM | 0.7298 | 0.3961 | 0.8119 | 0.5325 | 0.8331 | 164 из 202 | 250 |

kNN осторожен: называет хорошими 121 вино из 1066, из них верно 73 (Precision 0.6033), но пропускает 129 хороших, Recall всего 0.3614. SVM с весами `balanced` называет хорошими 414 вин, находит 164 хороших (Recall **0.8119**), но 250 раз ошибается (Precision 0.3961). Поэтому у SVM самая низкая из сильных моделей Accuracy (0.7298, ниже константы 0.8105), но лучший F1 (**0.5325** против 0.4520). Для задачи «отобрать кандидатов в хорошие вина» полнота важнее, и SVM здесь существенно полезнее, а Accuracy вводит в заблуждение.

**Остальные.** ComplementNB тоже ловит много хороших вин (Recall 0.7327), но ценой Precision 0.2726 и Accuracy 0.5788; его AUC-ROC низкий (0.7164), поэтому это просто сдвиг порога, а не хорошее ранжирование. LDA (Recall 0.3416, F1 0.4207) и дерево (Recall 0.2327, F1 0.3322) консервативны. GaussianNB - сбалансированный, но слабый (F1 0.4668, AUC 0.7581). BernoulliNB находит лишь 36 хороших вин из 202 (Recall 0.1782).

**Оптимизм кросс-валидации.** На тесте AUC-ROC у всех моделей ниже CV на 0.013-0.030: kNN -0.0136, SVM -0.0127, LDA -0.0171, дерево -0.0174, GaussianNB -0.0300, BernoulliNB -0.0143, наивные мультиномиальные -0.0250. Порядок методов по CV и по тесту в целом совпал: наверху kNN и SVM, затем LDA и дерево. Сильнее всех просел GaussianNB (-0.0300). Потери одного порядка у всех говорят об умеренном оптимизме выбора по кросс-валидации, а не о поломке конкретной модели.

**Рекомендация: SVM.** Применяя заранее зафиксированное правило, получаем: по AUC-ROC лидирует kNN, но SVM в пределах 0.01 от него; внутри этой полосы по Recall и F1 побеждает SVM (Recall 0.8119, F1 0.5325). Кроме того, SVM компактнее kNN в эксплуатации: ему не нужно хранить и перебирать все 4263 обучающих объекта при каждом запросе, достаточно опорных векторов. **Рекомендуемый метод - SVM** (`rbf`, `C=10`, `gamma=0.01`, `class_weight="balanced"`, `StandardScaler`). Если важнее точность названных хороших вин, чем их полнота, разумной заменой будет kNN: он выдаёт лишь 48 ложных тревог, но пропускает почти две трети хороших вин.

## 7. Нейронная сеть на TensorFlow

Задача та же: по 12 признакам (11 физико-химических и `is_red`) отличить хорошее вино (`quality >= 7`) от обычного. Порог решения - 0.5.

### Архитектура

Базовая модель - полносвязный многослойный перцептрон `tf.keras.Sequential`:

1. `Input(shape=(12,))` - 12 стандартизованных признаков.
2. Скрытые слои `Dense`: число слоёв и ширина первого слоя перебираются. Каждый следующий скрытый слой вдвое уже предыдущего, но не уже 8 нейронов. После первого скрытого слоя стоит `Dropout`.
3. Выход `Dense(1, activation="sigmoid")` - вероятность хорошего вина.

Выбранная в итоге сеть: `Dense(64, ReLU)` - `Dropout(0.2)` - `Dense(32, ReLU)` - `Dense(1, sigmoid)`, всего 2945 обучаемых параметров (\(12\cdot 64+64 = 832\), \(64\cdot 32+32 = 2080\), \(32+1 = 33\)).

Функции активации скрытых слоёв сравниваются отдельно: ReLU, tanh и ELU. На выходе сигмоида нужна вероятность из \([0, 1]\) для бинарной кросс-энтропии. Функция потерь - `binary_crossentropy`, оптимизатор - `Adam`. Keras пишет `loss` и `accuracy` для обучения и для проверки, TensorBoard забирает обе величины.

Сеть линейно комбинирует признаки, поэтому перед ней стоит `StandardScaler`, обученный только на той части выборки, которую видит данный прогон. Тест из раздела 6 в подбор не входит.

### Схема подбора

Обучающие 4263 объекта делятся ещё раз, стратифицированно: 3410 на обучение сети и 853 на проверку (из них около 162 хороших). Гиперпараметры подбираются **по одному**, от самого влиятельного к менее влиятельному, начиная от базовой точки (2 слоя, 64 нейрона, ReLU, `lr=1e-3`, batch 32, 50 эпох, dropout 0.2):

1. скорость обучения (\(10^{-4}\), \(10^{-3}\), \(10^{-2}\));
2. число нейронов в первом слое (16, 64, 128);
3. число скрытых слоёв (1, 2, 3);
4. функция активации (ReLU, tanh, ELU);
5. размер батча (16, 32, 64);
6. число эпох (20, 50, 100);
7. dropout (0, 0.2, 0.5).

После каждого шага выбирается значение с лучшим проверочным AUC-ROC (с допуском 0.005 на шум проверки), и следующий параметр перебирается уже вокруг найденной точки. Всего запущено **15 различных конфигураций**. Если сеть учится на несбалансированной выборке без поправки, она склонна отвечать «обычное», поэтому на финальном обучении сравниваются два варианта: с весами классов и без.

In [7]:
nn_runs = load("нейросеть_подбор.csv")
cols = ["Прогон", "units", "learning_rate", "dropout", "n_layers", "activation", "batch_size", "epochs", "val_auc", "val_recall", "gap_acc", "min_val_loss", "min_val_epoch", "val_loss_last"]
display(Markdown("### Все 15 прогонов подбора (проверочная часть: 853 объекта)"))
display(nn_runs[cols].set_index("Прогон"))

best = nn_runs.sort_values("val_auc", ascending=False).iloc[0]
print("Лучший проверочный AUC-ROC:", round(float(best["val_auc"]), 4), "-", best["Прогон"])

### Все 15 прогонов подбора (проверочная часть: 853 объекта)

,units,learning_rate,dropout,n_layers,activation,batch_size,epochs,val_auc,val_recall,gap_acc,min_val_loss,min_val_epoch,val_loss_last
Прогон,,,,,,,,,,,,,
L2_U64_relu_lr1e-04_d0.2_bs32_ep50,64,0.0001,0.2000,2,relu,32,50,0.8165,0.7160,0.0081,0.4891,47,0.4902
L2_U64_relu_lr1e-03_d0.2_bs32_ep50,64,0.0010,0.2000,2,relu,32,50,0.8320,0.7037,0.0271,0.4492,39,0.4502
L2_U64_relu_lr1e-02_d0.2_bs32_ep50,64,0.0100,0.2000,2,relu,32,50,0.8341,0.6173,0.0280,0.4382,38,0.4426
L2_U16_relu_lr1e-02_d0.2_bs32_ep50,16,0.0100,0.2000,2,relu,32,50,0.8119,0.6975,0.0110,0.4497,25,0.4617
L2_U128_relu_lr1e-02_d0.2_bs32_ep50,128,0.0100,0.2000,2,relu,32,50,0.8171,0.6173,0.0488,0.4371,4,0.4946
L1_U64_relu_lr1e-02_d0.2_bs32_ep50,64,0.0100,0.2000,1,relu,32,50,0.8237,0.7469,0.0198,0.4376,16,0.4803
L3_U64_relu_lr1e-02_d0.2_bs32_ep50,64,0.0100,0.2000,3,relu,32,50,0.8095,0.6481,0.0239,0.4378,6,0.4815
L2_U64_tanh_lr1e-02_d0.2_bs32_ep50,64,0.0100,0.2000,2,tanh,32,50,0.8254,0.6605,0.0166,0.4230,49,0.4343
L2_U64_elu_lr1e-02_d0.2_bs32_ep50,64,0.0100,0.2000,2,elu,32,50,0.8112,0.6481,-0.0010,0.4139,48,0.4441


Лучший проверочный AUC-ROC: 0.8341 - L2_U64_relu_lr1e-02_d0.2_bs32_ep50


### Мониторинг обучения через TensorBoard

Журнал каждого прогона пишется callback `tf.keras.callbacks.TensorBoard` в папку `C:\lr2_wine_tensorboard` (путь только из ASCII: в имени пользователя Windows кириллица, а TensorFlow на таком пути не создаёт event-файлы). Панель запускается командой

```
tensorboard --logdir "C:\lr2_wine_tensorboard"
```

и открывается по адресу `http://localhost:6006`. Вкладка Scalars показывает `epoch_loss` и `epoch_accuracy` для `train` и `validation`. Ниже те же ряды, прочитанные из event-файлов и построенные в matplotlib, чтобы графики были видны без запуска TensorBoard.

### Графики обучения и валидации

**Скорость обучения**

![TensorBoard: скорость обучения](figures/nn_tb_01_скорость_обучения.png)

**Число нейронов**

![TensorBoard: число нейронов](figures/nn_tb_02_число_нейронов.png)

**Число скрытых слоёв**

![TensorBoard: число слоёв](figures/nn_tb_03_число_слоёв.png)

**Функции активации**

![TensorBoard: активации](figures/nn_tb_04_активация.png)

**Размер батча**

![TensorBoard: размер батча](figures/nn_tb_05_размер_батча.png)

**Число эпох**

![TensorBoard: число эпох](figures/nn_tb_06_число_эпох.png)

**Dropout**

![TensorBoard: dropout](figures/nn_tb_07_dropout.png)

### Снимки интерфейса TensorBoard

Снимки сняты при запущенной команде `tensorboard --logdir "C:\lr2_wine_tensorboard"` (вкладка Scalars, сглаживание 0.6). В левой панели видны имена прогонов; на графиках — `epoch_accuracy`, `epoch_learning_rate` и `epoch_loss`.

**Обзор всех экспериментов** (полный журнал):

![TensorBoard UI: обзор](figures/tb_ui_scalars_overview.png)

**Скорость обучения** (фильтр `L2_U64_relu_lr*`): сравнение `1e-4`, `1e-3` и `1e-2`:

![TensorBoard UI: скорость обучения](figures/tb_ui_scalars_lr.png)

**Число скрытых слоёв** (один / два / три слоя при одинаковых остальных гиперпараметрах):

![TensorBoard UI: число слоёв](figures/tb_ui_scalars_layers.png)

**Функции активации** (ReLU, tanh, ELU):

![TensorBoard UI: активации](figures/tb_ui_scalars_activation.png)

**Итоговые сети** `final_balanced` и `final_plain`:

![TensorBoard UI: итоговые сети](figures/tb_ui_scalars_final.png)


### Вывод по экспериментам с гиперпараметрами

Внутренняя проверка - 853 объекта, из них около 162 хороших. Сдвиг AUC-ROC в 0.005 здесь того же порядка, что шум, поэтому близкие значения не объявляются победой. Смотреть нужно и на форму кривых: ниже сопоставлены `val_auc` из таблицы и поведение `val_loss`.

#### Скорость обучения

Скорость \(10^{-4}\) за 50 эпох не доучивается: validation loss падает с 0.7407 на первой эпохе до 0.4902 на последней, минимум приходится на 47-ю эпоху, а разрыв Accuracy обучения и проверки 0.008. Это не лучшее обобщение, а слишком короткий шаг: сеть просто не успела выучить зависимость. `val_auc` 0.8165. Скорость \(10^{-3}\) даёт 0.8320, validation loss 0.5359 → 0.4502 с минимумом около 39-й эпохи. Скорость \(10^{-2}\) даёт лучший **val_auc 0.8341**, validation loss 0.4585 → 0.4426 (минимум 0.4382 на 38-й эпохе), но Recall при пороге 0.5 здесь ниже (0.6173 против 0.7037 и 0.716): быстрая сеть увереннее отвечает «обычное». Выбрано \(10^{-2}\) по AUC-ROC.

#### Число нейронов

16 нейронов - слишком мало: `val_auc` 0.8119 при `val_loss` 0.4617. 64 нейрона - лучшая точка (0.8341). 128 нейронов уже хуже (0.8171): разрыв Accuracy вырос до 0.0488, минимум validation loss (0.4371) достигается уже на 4-й эпохе, к 50-й она вырастает до 0.4946. Это переобучение: широкая сеть быстро запоминает обучающие объекты.

#### Число слоёв

Один слой даёт 0.8237 (Recall 0.7469), два - 0.8341, три - 0.8095. Три слоя хуже и по AUC-ROC, и по validation loss (0.4815): для 12 признаков и 3410 объектов глубина больше двух только добавляет параметры. Выбраны два слоя.

#### Функция активации

ReLU - 0.8341, tanh - 0.8254, ELU - 0.8112. Интересно, что у tanh самая низкая validation loss (0.4343) и самая высокая Accuracy проверки (0.789), но ранжирование по AUC-ROC хуже на 0.009, то есть больше допуска 0.005. Выбран ReLU.

#### Размер батча

Batch 32 - 0.8341, batch 16 - 0.8177, batch 64 - 0.8203. Мелкий батч даёт шумные градиенты и больший разрыв Accuracy (0.0403), крупный - validation loss, растущий к концу (0.4995 при минимуме 0.4377). Среднее значение 32 - разумный компромисс.

#### Число эпох

20 эпох - 0.8294 (разрыв Accuracy -0.0007: сеть ещё не переобучилась), 50 - 0.8341, 100 - 0.8207. После 100 эпох разрыв Accuracy 0.0541, а validation loss 0.5056 при минимуме 0.4382 на 38-й эпохе: сеть, обученная дольше, ухудшает проверку. 20 эпох укладывается в допуск 0.005, но выбрано 50 как лучшая точка по AUC-ROC.

#### Dropout

Без dropout `val_auc` 0.8118, разрыв Accuracy 0.087, validation loss к концу 0.6863 при минимуме 0.4285 на 9-й эпохе: сеть сильно переобучается после девятой эпохи. С `dropout=0.2` - 0.8341. С `dropout=0.5` - 0.8166 (разрыв всего 0.0054, но слишком сильная регуляризация недоучивает сеть). Выбран 0.2.

**Итог подбора:** `lr=0.01`, 64 нейрона, 2 слоя, ReLU, batch 32, 50 эпох, `dropout=0.2`; проверочный AUC-ROC **0.8341**.

### Одна оценка на отложенном тесте

Выбранная сеть обучается заново на всех 4263 обучающих объектах с теми слоями, активацией, числом эпох и размером батча, которые оставила внутренняя проверка. Масштабировщик тоже пересчитывается по всей обучающей части: иначе тест сравнивал бы сеть, которая видела меньше объектов, чем SVM и kNN.

Обучаются два варианта с одинаковыми начальными весами: **с весами классов** (вес хорошего вина около 2.6, обычного около 0.6) и **без весов**. Веса классов в Keras играют ту же роль, что `class_weight="balanced"` у SVM: сдвигают порог решения в пользу редкого класса. Тестовые 1066 объектов не участвуют ни в одном градиентном шаге.

In [8]:
nn_test = load("нейросеть_тест.csv", index_col=0)
display(Markdown("### Сеть и классические методы на тесте"))
display(nn_test)

# Матрицы ошибок по Precision, Recall и размерам классов теста.
rows = []
for name, r in nn_test.iterrows():
    tp = int(round(r["Recall"] * N_POS))
    fp = int(round(tp / r["Precision"])) - tp
    rows.append({"Модель": name, "TP": tp, "FN": N_POS - tp, "FP": fp, "TN": N_NEG - fp})
display(Markdown("### Матрицы ошибок (восстановлены по метрикам)"))
display(pd.DataFrame(rows).set_index("Модель"))

### Сеть и классические методы на тесте

,Accuracy,Precision,Recall,F1-Score,AUC-ROC
"Сеть, веса классов",0.7505,0.4167,0.7921,0.5461,0.8371
"Сеть, без весов",0.8340,0.6168,0.3267,0.4272,0.8348
kNN (лучший классический по AUC),0.8340,0.6033,0.3614,0.4520,0.8345
SVM (рекомендованный),0.7298,0.3961,0.8119,0.5325,0.8331


### Матрицы ошибок (восстановлены по метрикам)

,TP,FN,FP,TN
Модель,,,,
"Сеть, веса классов",160,42,224,640
"Сеть, без весов",66,136,41,823
kNN (лучший классический по AUC),73,129,48,816
SVM (рекомендованный),164,38,250,614


### Матрицы ошибок и ROC-кривые сети

Первый график - матрицы ошибок двух вариантов сети (с весами классов и без), второй - ROC-кривые сети рядом с классическими методами.

![Матрицы ошибок нейронной сети](figures/nn_tb_08_матрицы_ошибок.png)

![ROC: нейронная сеть и классические методы](figures/nn_tb_09_roc_сеть_и_классика.png)

### Вывод по сети на тесте

Выбранная архитектура: вход 12 признаков, `Dense(64, ReLU)`, `Dropout(0.2)`, `Dense(32, ReLU)`, выход `Dense(1, sigmoid)`, потери `binary_crossentropy`, оптимизатор Adam с \(10^{-2}\), 50 эпох, пакет 32.

**Сеть с весами классов** даёт на отложенном тесте Accuracy **0.7505**, Precision 0.4167, Recall **0.7921**, F1 **0.5461**, AUC-ROC **0.8371**. Матрица: 160 найденных хороших вин из 202 (42 пропуска), 224 ложных тревоги, 640 верно отброшенных обычных. **Та же сеть без весов**: Accuracy 0.8340, Precision 0.6168, Recall 0.3267, F1 0.4272, AUC-ROC 0.8348 (66 найдено, 136 пропусков, 41 ложная тревога).

Веса классов почти не сдвинули ранжирование: AUC-ROC 0.8371 против 0.8348, то есть разница 0.0023. Главное, что они сделали, - переместили рабочую точку по ROC-кривой: Recall вырос с 0.3267 до 0.7921, а Precision упала с 0.6168 до 0.4167. Это та же картина, что у SVM с `balanced`.

**Сравнение с классикой на тесте:**

| Модель | Accuracy | Precision | Recall | F1 | AUC-ROC |
|---|---|---|---|---|---|
| Сеть, веса классов | 0.7505 | 0.4167 | 0.7921 | 0.5461 | 0.8371 |
| Сеть, без весов | 0.8340 | 0.6168 | 0.3267 | 0.4272 | 0.8348 |
| kNN (лучший классический по AUC) | 0.8340 | 0.6033 | 0.3614 | 0.4520 | 0.8345 |
| SVM (рекомендованный) | 0.7298 | 0.3961 | 0.8119 | 0.5325 | 0.8331 |

Сеть с весами и SVM практически эквивалентны. AUC-ROC сети выше на 0.0040 (0.8371 против 0.8331), F1 выше на 0.0136 (0.5461 против 0.5325), а Recall ниже: 160 найденных хороших вин против 164 у SVM, то есть на 4 вина, или 0.0198 по Recall. Все три разрыва - порядка нескольких вин из 202 или нескольких тысячных AUC-ROC, и не выходят за шум одного разбиения. Сеть без весов и kNN тоже идут почти вровень (AUC-ROC 0.8348 и 0.8345; Accuracy у обоих 0.8340) - обе осторожные и находят мало хороших вин.

**Итог по сети: она не обогнала SVM по существу.** Сложная модель с 2945 параметрами, подбором архитектуры по 15 прогонам и необходимостью TensorFlow дала тот же уровень, что SVM с одним `fit` в scikit-learn: потолок задан шумом оценок дегустаторов и перекрытием классов, а не ёмкостью модели. **Рекомендация остаётся за SVM** по простоте: он воспроизводится быстрее, не требует TensorFlow и даёт тот же Recall и F1 в пределах шума. Сеть оправдана, только если нужен лишний ~0.004 AUC-ROC или планируется расширять набор признаков.

## 8. Заключение

В работе решена бинарная задача: по 12 признакам (11 физико-химических показателей и тип вина) определить, получит ли вино оценку `quality >= 7`. После очистки (удалено 1168 дубликатов) осталось 5329 объектов, из них 1011 хороших (18.97%). Разбиение 4263 / 1066 стратифицировано, тест не участвовал ни в одном подборе.

Самый сильный одиночный признак - спирт (Cohen's d = +1.155), затем плотность (-0.791), хлориды (-0.478) и летучая кислотность (-0.411): хорошие вина крепче, менее плотные и менее «дефектные».

Настройка гиперпараметров дала наибольший прирост CV AUC-ROC у дерева (+0.1535, 0.6494 → 0.8029), kNN (+0.0474, до 0.8481) и SVM (+0.0252, до 0.8458). У мультиномиальных вариантов прироста нет, у LDA стартовая точка уже была на верхней полке (0.8276).

На тесте по AUC-ROC лидирует **kNN (0.8345)**, но **SVM (0.8331)** отстаёт всего на 0.0014 и при этом находит **81.2% хороших вин** (Recall 0.8119, F1 0.5325) против 36.1% у kNN (F1 0.4520). Ценой выступает низкая Accuracy (0.7298, ниже константы 0.8105) из-за 250 ложных тревог. Это подтверждает: на несбалансированных данных Accuracy нельзя читать отдельно. **Рекомендуемый метод - SVM.**

Нейронная сеть (2 слоя, 64 нейрона, ReLU, dropout 0.2, `lr=0.01`) с весами классов получила AUC-ROC 0.8371, Recall 0.7921, F1 0.5461. Это на уровне SVM; потолок задаёт шум оценок вина, поэтому сложность модели ничего не добавляет. Выбор остаётся за SVM.

In [9]:
print("Полный пересчёт: py -3.12 lab2_classification.py")
print("Исполняемый конвейер (данные, сетки, графики, нейросеть, TensorBoard) находится в файле lab2_classification.py;")
print("он пересоздаёт все графики в figures/ и таблицы в results/, которые читает и показывает этот ноутбук.")
print("Панель TensorBoard:", f'tensorboard --logdir "{LOG_ROOT}"')

Полный пересчёт: py -3.12 lab2_classification.py
Исполняемый конвейер (данные, сетки, графики, нейросеть, TensorBoard) находится в файле lab2_classification.py;
он пересоздаёт все графики в figures/ и таблицы в results/, которые читает и показывает этот ноутбук.
Панель TensorBoard: tensorboard --logdir "C:\lr2_wine_tensorboard"
